In [1]:
from IPython.display import display, HTML
from fasthtml.common import *
from fasthtml.jupyter import *

def create_server(app,rt):
    if IN_NOTEBOOK:
        for port in range(8000,8030):   
            if 'server' in globals():
                print(f"Server already running on port {globals()['server'].port} - stopping it")
                globals()['server'].stop()
            if is_port_free(port):
                server = JupyUvi(app, port=port)

                def HShow(comp, app):
                    @app.get('/')
                    def get(): return comp
                    display(HTML(f'<a href="http://localhost:{port}/" target="_blank">Open in new tab</a>'))
                    return HTMX("/",port=port, iframe_height="300px")
                print(f"Starter server on port {port}")
                Show = partial(HShow, app=app)
                return app, rt, server, HShow, Show


In [2]:
from dataclasses import dataclass
import json, hashlib

@dataclass
class Component:
    """Base component class that handles template and props"""
    template: str
    name: str = None
    
    def __post_init__(self):
        if not self.name:
            hash_obj = hashlib.md5(self.template.encode())
            self.name = f"PyComponent_{hash_obj.hexdigest()[:8]}"
    
    def __call__(self, *children, **props):
        return Div(
            *children,
            data_component=self.name,
            data_props=json.dumps({
                **props,
                '_childComponents': [
                    {'name': c['data_component'], 'props': json.loads(c['data_props'])}
                    for c in children if not isinstance(c, str)
                ]
            }),
            cls=f"py-component {self.name}"
        )

class React:
    """React framework integration"""
    def __init__(self):
        self.components = {}
        
    def component(self, template):
        """Decorator to create React components"""
        def wrapper(cls):
            comp = Component(template, cls.__name__)
            self.components[comp.name] = template
            return comp
        return wrapper
    
    def get_runtime(self):
        """Generate runtime JS for all registered components"""
        components_js = "\n".join(
            f"""window['{name}'] = {template};"""
            for name, template in self.components.items()
        )
        
        return Script(f"""
            {components_js}
            
            function createReactElement(name, props) {{
                const Component = window[name];
                if (!Component) return null;
                
                // Handle child components
                if (props._childComponents) {{
                    props.children = props._childComponents.map(child => 
                        createReactElement(child.name, child.props)
                    );
                    delete props._childComponents;
                }}
                
                return React.createElement(Component, props);
            }}
            
            // Hydrate components after DOM loads
            htmx.onLoad(() => {{
                document.querySelectorAll('.py-component').forEach(el => {{
                    const name = el.dataset.component;
                    const props = JSON.parse(el.dataset.props || '{{}}');
                    
                    // Create element tree
                    const element = createReactElement(name, props);
                    if (!element) return;
                    
                    // Render
                    ReactDOM.render(element, el);
                }});
            }});
        """)

def get_framework_headers():
    """Return required React CDN scripts"""
    return (
        Script(src="https://unpkg.com/react@17/umd/react.production.min.js"),
        Script(src="https://unpkg.com/react-dom@17/umd/react-dom.production.min.js"),
    )

In [3]:

react = React()

@react.component("""
    function Button({children, color="blue", onClick}) {
        const style = {
            padding: '8px 16px',
            backgroundColor: color,
            color: 'white',
            border: 'none',
            borderRadius: '4px',
            cursor: 'pointer',
            margin: '4px'
        };
        return React.createElement('button', {
            style,
            onClick: () => onClick && window[onClick]()
        }, children);
    }
""")
class Button: pass

@react.component("""
    function Card({children, title}) {
        const style = {
            padding: '16px',
            border: '1px solid #eee',
            borderRadius: '8px',
            margin: '8px'
        };
        const titleStyle = {
            margin: '0 0 16px 0'
        };
        return React.createElement('div', {style}, [
            React.createElement('h3', {style: titleStyle, key: 'title'}, title),
            children
        ]);
    }
""")
class Card: pass

In [4]:
app, rt = fast_app(pico=False,
    hdrs=(*get_framework_headers(), react.get_runtime())
)
app, rt, server, HShow, Show = create_server(app, rt)

Starter server on port 8000


In [5]:
client_functions = Script("""
    function showAlert() {
        alert('Button clicked!');
    }
""")

Show(Titled("React Components Demo",
        client_functions,
        Card(
            Button("Click Me!", 
                   color="purple", 
                   onClick="showAlert"),
            Button("Another Button", 
                   color="teal", 
                   onClick="showAlert"),
            title="Demo Card"
        )
    )
    )

TypeError: tuple indices must be integers or slices, not str